# Dataset Visualisation — wi29 Western Sahara (TSPLIB95)

Run cells one by one. Each cell is independent — if Null is too slow, skip it and run Min-edge + MST only.

## Cell 1 — Imports and path setup

In [ ]:
import sys, os, time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

# Path setup for this exact folder structure:
# A-start-heuristics/
#   code/phase1_foundation, phase2_astar, phase3_heuristics
#   experiments/phase4_experiments  ← this notebook

THIS_DIR = os.path.abspath('')
EXPS_DIR = os.path.dirname(THIS_DIR)
ROOT_DIR = os.path.dirname(EXPS_DIR)
CODE_DIR = os.path.join(ROOT_DIR, 'code')

sys.path.insert(0, os.path.join(CODE_DIR, 'phase1_foundation'))
sys.path.insert(0, os.path.join(CODE_DIR, 'phase2_astar'))
sys.path.insert(0, os.path.join(CODE_DIR, 'phase3_heuristics'))

DATA_DIR    = os.path.join(ROOT_DIR, 'data')
RESULTS_DIR = os.path.join(ROOT_DIR, 'results', 'charts')
os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

from tsp import City, build_distance_matrix, load_tsplib, download_wi29, generate_random_cities
from state import make_start_state
from baseline import brute_force_tsp
from astar import astar_tsp, null_heuristic, min_edge_heuristic
from heuristic_mst import mst_heuristic

print('All imports OK')
print(f'ROOT_DIR = {ROOT_DIR}')
print(f'CODE_DIR = {CODE_DIR}')

## Cell 2 — Load wi29 dataset

In [ ]:
tsp_path = download_wi29(save_dir=DATA_DIR)
cities   = load_tsplib(tsp_path)
dist     = build_distance_matrix(cities)

print(f'Cities loaded : {len(cities)}')
print(f'X range       : {min(c.x for c in cities):.2f} – {max(c.x for c in cities):.2f}')
print(f'Y range       : {min(c.y for c in cities):.2f} – {max(c.y for c in cities):.2f}')

# Quick scatter to confirm layout
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter([c.x for c in cities], [c.y for c in cities],
           s=60, color='#0D1B2A', zorder=3, edgecolors='white', linewidths=0.8)
ax.scatter([cities[0].x], [cities[0].y],
           s=120, color='#F4A324', zorder=4, edgecolors='white', linewidths=1.2,
           label='Start city')
for i, c in enumerate(cities):
    ax.annotate(str(i+1), (c.x, c.y), textcoords='offset points',
                xytext=(5,5), fontsize=7, color='#444444')
ax.set_title('wi29 — 29 cities, Western Sahara (TSPLIB95)', fontsize=11, fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])
ax.legend(fontsize=9)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'wi29_cities.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Cities plotted — no tours yet.')

## Cell 3 — Run MST heuristic (fast — run this first)

In [ ]:
print('Running MST heuristic on wi29...')
t0    = time.perf_counter()
r_mst = astar_tsp(dist, mst_heuristic)
t_mst = (time.perf_counter() - t0) * 1000

print(f'  Cost          : {r_mst.cost:.4f}')
print(f'  Nodes expanded: {r_mst.nodes_expanded:,}')
print(f'  Runtime       : {t_mst:.1f} ms')
print(f'  Tour          : {" → ".join(str(i) for i in r_mst.path)}')

## Cell 4 — Run Min-edge heuristic (medium speed)

In [ ]:
print('Running Min-edge heuristic on wi29...')
t0   = time.perf_counter()
r_me = astar_tsp(dist, min_edge_heuristic)
t_me = (time.perf_counter() - t0) * 1000

print(f'  Cost          : {r_me.cost:.4f}')
print(f'  Nodes expanded: {r_me.nodes_expanded:,}')
print(f'  Runtime       : {t_me:.1f} ms')

## Cell 5 — Run Null heuristic (SLOW — optional, skip if taking too long)

> **Note:** Null on 29 cities can take 5–15 min on a slow machine.
> If it's too slow, skip this cell and the tables will use 'N/A' for Null on wi29.
> That actually strengthens your failure argument.

In [ ]:
RUN_NULL = False   # ← Set True if you want to wait for Null on wi29

if RUN_NULL:
    print('Running Null (UCS) on wi29 — this will take several minutes...')
    t0     = time.perf_counter()
    r_null = astar_tsp(dist, null_heuristic)
    t_null = (time.perf_counter() - t0) * 1000
    print(f'  Cost          : {r_null.cost:.4f}')
    print(f'  Nodes expanded: {r_null.nodes_expanded:,}')
    print(f'  Runtime       : {t_null:.1f} ms')
else:
    r_null = None
    t_null = None
    print('Null skipped on wi29 — too slow (exponential wall in action).')
    print('This is your striking failure evidence.')

## Cell 6 — Plot tour maps side by side

In [ ]:
COLORS = {
    'Null (UCS)' : '#888780',
    'Min-edge'   : '#378ADD',
    'MST'        : '#1D9E75',
    'start'      : '#F4A324',
}

def draw_tour(ax, cities, tour, color, title, cost, nodes, time_ms, optimal):
    for i in range(len(tour)):
        a = cities[tour[i]]
        b = cities[tour[(i+1) % len(tour)]]
        ax.plot([a.x, b.x], [a.y, b.y],
                color=color, linewidth=1.4, alpha=0.75, zorder=1)
    xs = [c.x for c in cities]
    ys = [c.y for c in cities]
    ax.scatter(xs, ys, s=40, color=color, zorder=3,
               edgecolors='white', linewidths=0.8)
    ax.scatter([cities[0].x], [cities[0].y],
               s=120, color=COLORS['start'], zorder=4,
               edgecolors='white', linewidths=1.2)
    for i, c in enumerate(cities):
        ax.annotate(str(i+1), (c.x, c.y),
                    textcoords='offset points', xytext=(4,4),
                    fontsize=6.5, color='#444444')
    opt_col = '#1D9E75' if optimal else '#E85D4A'
    opt_str = 'Optimal' if optimal else 'Sub-optimal'
    ax.set_title(title, fontsize=11, fontweight='bold', pad=4)
    ax.set_xlabel(
        f'Cost: {cost:.2f}  |  Nodes: {nodes:,}  |  '
        f'Time: {time_ms:.1f}ms  |  {opt_str}',
        fontsize=8.5, color=opt_col)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect('equal')
    for spine in ax.spines.values():
        spine.set_edgecolor('#DDDDDD')
        spine.set_linewidth(0.8)

opt_cost = r_mst.cost

# Build list of results to plot (skip Null if not run)
to_plot = []
if r_null:
    to_plot.append(('Null (UCS)', r_null, t_null))
to_plot.append(('Min-edge', r_me, t_me))
to_plot.append(('MST',      r_mst, t_mst))

fig, axes = plt.subplots(1, len(to_plot), figsize=(7 * len(to_plot), 6))
if len(to_plot) == 1:
    axes = [axes]
fig.patch.set_facecolor('white')

for ax, (h_name, res, t_ms) in zip(axes, to_plot):
    draw_tour(
        ax      = ax,
        cities  = cities,
        tour    = res.path,
        color   = COLORS[h_name],
        title   = h_name,
        cost    = res.cost,
        nodes   = res.nodes_expanded,
        time_ms = t_ms,
        optimal = abs(res.cost - opt_cost) < 1e-6,
    )

fig.suptitle('A* Heuristic Tours — wi29 Western Sahara (TSPLIB95)',
             fontsize=13, fontweight='bold', y=1.02)

legend_patches = [
    mpatches.Patch(color=COLORS[h], label=h)
    for h, _, _ in to_plot
] + [mpatches.Patch(color=COLORS['start'], label='Start city')]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=len(legend_patches), fontsize=9, frameon=False,
           bbox_to_anchor=(0.5, -0.04))

plt.tight_layout()
tour_path = os.path.join(RESULTS_DIR, 'wi29_tours.png')
plt.savefig(tour_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {tour_path}')

## Cell 7 — Breakdown table (wi29 results)

In [ ]:
admissible = {'Null (UCS)': 'Yes', 'Min-edge': 'Yes', 'MST': 'Yes'}
consistent = {'Null (UCS)': 'Yes', 'Min-edge': 'No',  'MST': 'Yes'}

# Reference node count for savings calculation
null_nodes = r_null.nodes_expanded if r_null else None
me_nodes   = r_me.nodes_expanded
mst_nodes  = r_mst.nodes_expanded

rows = []
for h_name, res, t_ms in to_plot:
    is_opt = abs(res.cost - opt_cost) < 1e-6
    if null_nodes:
        saving = f"{100*(null_nodes - res.nodes_expanded)/null_nodes:.1f}%"
    else:
        saving = 'N/A (Null not run)' if h_name != 'Null (UCS)' else '—'
    rows.append([
        h_name,
        f"{res.cost:.4f}",
        f"{res.nodes_expanded:,}",
        f"{t_ms:.1f} ms",
        saving,
        'Yes' if is_opt else 'No',
        admissible[h_name],
        consistent[h_name],
    ])

col_labels = ['Heuristic', 'Tour cost', 'Nodes expanded', 'Runtime',
              'Nodes saved vs Null', 'Optimal?', 'Admissible?', 'Consistent?']

fig, ax = plt.subplots(figsize=(16, 1.8 + 0.5 * len(rows)))
fig.patch.set_facecolor('white')
ax.axis('off')
ax.set_title('Heuristic breakdown — wi29 Western Sahara (29 cities)',
             fontsize=11, fontweight='bold', pad=8)

cell_colors = []
for i, row in enumerate(rows):
    rc = ['#F7F9FB' if i % 2 == 0 else '#FFFFFF'] * len(row)
    for j, v in enumerate(row):
        if v == 'Yes': rc[j] = '#EAF6F2'
        elif v == 'No': rc[j] = '#FCEAEA'
    cell_colors.append(rc)

table = ax.table(
    cellText=rows, colLabels=col_labels,
    cellLoc='center', loc='center',
    cellColours=cell_colors,
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.0)

for j in range(len(col_labels)):
    cell = table[0, j]
    cell.set_facecolor('#0D1B2A')
    cell.set_text_props(color='white', fontweight='bold')
    cell.set_edgecolor('#DDDDDD')
for i, row in enumerate(rows):
    for j, val in enumerate(row):
        cell = table[i+1, j]
        cell.set_edgecolor('#DDDDDD')
        if val == 'Yes':
            cell.set_text_props(color='#0F6E56', fontweight='bold')
        elif val == 'No':
            cell.set_text_props(color='#A32D2D', fontweight='bold')

plt.tight_layout()
tbl_path = os.path.join(RESULTS_DIR, 'wi29_breakdown_table.png')
plt.savefig(tbl_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {tbl_path}')

## Cell 8 — Cross-size comparison table (random 5, 8, 10 vs wi29)

In [ ]:
sizes  = [5, 8, 10, len(cities)]
labels = ['5 cities (random)', '8 cities (random)',
          '10 cities (random)', f'{len(cities)} cities (wi29 benchmark)']

cross_rows = []
print('Computing cross-size comparison...\n')

for sz, lbl in zip(sizes, labels):
    if sz == len(cities):
        cl, dm = cities, dist
    else:
        cl = generate_random_cities(sz, seed=42)
        dm = build_distance_matrix(cl)

    rn = astar_tsp(dm, null_heuristic)
    rm = astar_tsp(dm, min_edge_heuristic)
    rs = astar_tsp(dm, mst_heuristic)

    if sz == len(cities) and not r_null:
        # Use known MST result, skip Null re-run
        rn = None

    saving = '—'
    if rn and rs:
        saving = f"{100*(rn.nodes_expanded - rs.nodes_expanded)/rn.nodes_expanded:.1f}%"
    elif rs:
        saving = 'N/A (Null not run)'

    cross_rows.append([
        lbl,
        f"{rn.nodes_expanded:,}" if rn else 'Too slow',
        f"{rm.nodes_expanded:,}" if rm else 'N/A',
        f"{rs.nodes_expanded:,}" if rs else 'N/A',
        saving,
        f"{rs.cost:.4f}"         if rs else 'N/A',
    ])
    print(f"  {lbl:<32} "
          f"Null={rn.nodes_expanded:>10,}  " if rn else f"  {lbl:<32} Null={'Too slow':>10}  ",
          end='')
    print(f"MST={rs.nodes_expanded:>8,}  saving={saving}" if rs else 'MST=N/A')

cross_cols = ['Dataset', 'Null nodes', 'Min-edge nodes',
              'MST nodes', 'MST saving vs Null', 'MST cost']

fig, ax = plt.subplots(figsize=(16, 2.5 + 0.5 * len(cross_rows)))
fig.patch.set_facecolor('white')
ax.axis('off')
ax.set_title('Cross-dataset comparison — random instances vs wi29 benchmark',
             fontsize=11, fontweight='bold', pad=8)

cell_colors = [['#F7F9FB' if i % 2 == 0 else '#FFFFFF'] * len(cross_cols)
               for i in range(len(cross_rows))]

# Highlight wi29 row
cell_colors[-1] = ['#E6F1FB'] * len(cross_cols)

table2 = ax.table(
    cellText=cross_rows, colLabels=cross_cols,
    cellLoc='center', loc='center',
    cellColours=cell_colors,
)
table2.auto_set_font_size(False)
table2.set_fontsize(9)
table2.scale(1, 2.0)

for j in range(len(cross_cols)):
    cell = table2[0, j]
    cell.set_facecolor('#0D1B2A')
    cell.set_text_props(color='white', fontweight='bold')
    cell.set_edgecolor('#DDDDDD')
for i in range(len(cross_rows)):
    for j in range(len(cross_cols)):
        table2[i+1, j].set_edgecolor('#DDDDDD')

# Bold the wi29 row
for j in range(len(cross_cols)):
    table2[len(cross_rows), j].set_text_props(fontweight='bold', color='#0C447C')

plt.tight_layout()
cross_path = os.path.join(RESULTS_DIR, 'wi29_cross_comparison.png')
plt.savefig(cross_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {cross_path}')

## Cell 9 — Combined figure (all three panels — this is what goes in the slide)

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(3, len(to_plot),
                        figure=fig, hspace=0.55, wspace=0.3,
                        height_ratios=[2.2, 1.0, 1.0])

# Row 1 — tours
for col, (h_name, res, t_ms) in enumerate(to_plot):
    ax = fig.add_subplot(gs[0, col])
    draw_tour(
        ax=ax, cities=cities, tour=res.path,
        color=COLORS[h_name], title=h_name,
        cost=res.cost, nodes=res.nodes_expanded,
        time_ms=t_ms,
        optimal=abs(res.cost - opt_cost) < 1e-6,
    )

# Row 2 — breakdown table
ax2 = fig.add_subplot(gs[1, :])
ax2.axis('off')
ax2.set_title('Heuristic breakdown — wi29 Western Sahara (29 cities)',
              fontsize=10, fontweight='bold', pad=6)
cc2 = []
for i, row in enumerate(rows):
    rc = ['#F7F9FB' if i%2==0 else '#FFFFFF'] * len(row)
    for j,v in enumerate(row):
        if v=='Yes': rc[j]='#EAF6F2'
        elif v=='No': rc[j]='#FCEAEA'
    cc2.append(rc)
t2 = ax2.table(cellText=rows, colLabels=col_labels,
               cellLoc='center', loc='center', cellColours=cc2)
t2.auto_set_font_size(False); t2.set_fontsize(9); t2.scale(1, 1.7)
for j in range(len(col_labels)):
    t2[0,j].set_facecolor('#0D1B2A')
    t2[0,j].set_text_props(color='white', fontweight='bold')
    t2[0,j].set_edgecolor('#DDDDDD')
for i,row in enumerate(rows):
    for j,v in enumerate(row):
        t2[i+1,j].set_edgecolor('#DDDDDD')
        if v=='Yes': t2[i+1,j].set_text_props(color='#0F6E56',fontweight='bold')
        elif v=='No': t2[i+1,j].set_text_props(color='#A32D2D',fontweight='bold')

# Row 3 — cross comparison table
ax3 = fig.add_subplot(gs[2, :])
ax3.axis('off')
ax3.set_title('Cross-dataset comparison — random instances vs wi29 benchmark',
              fontsize=10, fontweight='bold', pad=6)
cc3 = [['#F7F9FB' if i%2==0 else '#FFFFFF']*len(cross_cols)
       for i in range(len(cross_rows))]
cc3[-1] = ['#E6F1FB']*len(cross_cols)
t3 = ax3.table(cellText=cross_rows, colLabels=cross_cols,
               cellLoc='center', loc='center', cellColours=cc3)
t3.auto_set_font_size(False); t3.set_fontsize(9); t3.scale(1, 1.7)
for j in range(len(cross_cols)):
    t3[0,j].set_facecolor('#0D1B2A')
    t3[0,j].set_text_props(color='white',fontweight='bold')
    t3[0,j].set_edgecolor('#DDDDDD')
for i in range(len(cross_rows)):
    for j in range(len(cross_cols)):
        t3[i+1,j].set_edgecolor('#DDDDDD')
for j in range(len(cross_cols)):
    t3[len(cross_rows),j].set_text_props(fontweight='bold',color='#0C447C')

fig.suptitle('A* Heuristic Comparison — wi29 Western Sahara (TSPLIB95)',
             fontsize=14, fontweight='bold', y=0.98, color='#0D1B2A')

lp = [mpatches.Patch(color=COLORS[h], label=h) for h,_,_ in to_plot]
lp += [mpatches.Patch(color=COLORS['start'], label='Start city')]
fig.legend(handles=lp, loc='lower center', ncol=len(lp),
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, 0.01))

combined_path = os.path.join(RESULTS_DIR, 'dataset_wi29.png')
plt.savefig(combined_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Combined figure saved → {combined_path}')
print('Ready for add_dataset_slide.py')